# DATAMYST AI
## Autonomous Data Forensics & Hidden Pattern Discovery Engine

An end-to-end exploratory investigation workflow for tabular data: quality auditing, anomaly detection, clustering, PCA, relationship discovery, visual analytics, and exportable findings.

**Run from top to bottom.** Set `DATA_PATH` to your CSV file, or leave it as `None` to use synthetic demonstration data.

**Responsible use:** anomalies are review candidates, not proof of fraud or errors. Correlation and mutual information do not prove causation. Clusters need domain validation. Synthetic data must not be presented as real-world evidence.

## 0. Configuration and imports
Edit the configuration below before running.

In [ ]:

# If needed, install packages:
# %pip install numpy pandas matplotlib seaborn scipy scikit-learn

import os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.feature_selection import mutual_info_regression

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH = None  # Example: "my_dataset.csv"
OUTPUT_DIR = Path("DATAMYST_AI_RESULTS")
ANOMALY_CONTAMINATION = 0.04
MAX_HEATMAP_FEATURES = 25
OUTPUT_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
print("DATAMYST AI ready. Results:", OUTPUT_DIR.resolve())


## 1. Load a dataset
The demo data deliberately includes missing values, duplicate records, and unusual observations to demonstrate the pipeline.

In [ ]:

def generate_demo_data(n=1800, seed=42):
    rng = np.random.default_rng(seed)
    region = rng.choice(["North","South","East","West"], n, p=[.25,.30,.20,.25])
    kind = rng.choice(["Standard","Premium","Enterprise"], n, p=[.55,.30,.15])
    age = rng.normal(38,12,n).clip(18,80)
    income = rng.normal(55000,18000,n).clip(12000,150000)
    activity = rng.gamma(4,8,n)
    satisfaction = rng.normal(7,1.8,n).clip(1,10)
    rf = pd.Series(region).map({"North":1.10,"South":1.25,"East":.85,"West":1.0}).to_numpy()
    tf = pd.Series(kind).map({"Standard":.8,"Premium":1.3,"Enterprise":1.8}).to_numpy()
    spending = (income*.12 + activity*120 + satisfaction*350)*rf*tf + rng.normal(0,800,n)
    unusual = rng.choice(n, max(1,int(n*.025)), replace=False)
    spending[unusual] *= rng.choice([.12,3.5,5.0],len(unusual))
    activity[unusual] *= rng.choice([.15,4.0],len(unusual))
    p = 1/(1+np.exp(-(.45*(6-satisfaction)+.025*(15-activity)+.000012*(30000-income))))
    churn = rng.binomial(1,np.clip(p,.02,.95))
    data = pd.DataFrame({
        "Record_ID":np.arange(10001,10001+n), "Age":age,
        "Annual_Income":income, "Monthly_Activity":activity,
        "Satisfaction":satisfaction, "Spending":spending,
        "Region":region, "Customer_Type":kind, "Churn":churn
    })
    for col in ["Annual_Income","Satisfaction","Monthly_Activity"]:
        idx = rng.choice(n,max(1,int(n*.035)),replace=False)
        data.loc[idx,col] = np.nan
    return pd.concat([data,data.iloc[:8]],ignore_index=True)

if DATA_PATH:
    if not Path(DATA_PATH).exists():
        raise FileNotFoundError(DATA_PATH)
    df_raw = pd.read_csv(DATA_PATH)
    data_source = str(DATA_PATH)
else:
    df_raw = generate_demo_data()
    data_source = "Synthetic demonstration dataset"

df_raw = df_raw.copy()
print("Source:",data_source,"| Shape:",df_raw.shape)
display(df_raw.head())


## 2. Data-quality audit
Inspect missingness, duplicates, data types, unique values, and constant columns before modeling.

In [ ]:

duplicate_count = int(df_raw.duplicated().sum())
audit = pd.DataFrame({
    "column":df_raw.columns,
    "dtype":[str(df_raw[c].dtype) for c in df_raw.columns],
    "missing_count":df_raw.isna().sum().values,
    "missing_pct":(df_raw.isna().mean().values*100).round(2),
    "unique_non_null":df_raw.nunique(dropna=True).values,
    "constant_or_empty":[df_raw[c].nunique(dropna=True)<=1 for c in df_raw.columns]
}).sort_values("missing_pct",ascending=False)

missing_ratio = float(df_raw.isna().to_numpy().mean())
duplicate_ratio = duplicate_count/max(len(df_raw),1)
constant_ratio = float(audit["constant_or_empty"].mean())
quality_score = float(np.clip(100-40*missing_ratio-30*duplicate_ratio-30*constant_ratio,0,100))

print("Rows:",len(df_raw),"Columns:",df_raw.shape[1])
print("Duplicate rows:",duplicate_count,"Missing cells:",int(df_raw.isna().sum().sum()))
print(f"Heuristic data-quality score: {quality_score:.1f}/100")
display(audit)
audit.to_csv(OUTPUT_DIR/"01_data_quality_audit.csv",index=False)


## 3. Feature preparation
Identifier columns are excluded. Numeric fields use median imputation and scaling; categorical fields use most-frequent imputation and one-hot encoding.

In [ ]:

id_names = {"id","record_id","customer_id","user_id","patient_id","transaction_id","row_id","uuid"}
identifier_cols = [c for c in df_raw.columns if c.lower() in id_names]
candidate_cols = [c for c in df_raw.columns if c not in identifier_cols and df_raw[c].nunique(dropna=True)>1]
numeric_cols = [c for c in candidate_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
categorical_cols = [c for c in candidate_cols if c not in numeric_cols]
if not candidate_cols:
    raise ValueError("No usable features remain.")

transformers = []
if numeric_cols:
    transformers.append(("numeric",Pipeline([
        ("imputer",SimpleImputer(strategy="median")),
        ("scaler",StandardScaler())
    ]),numeric_cols))
if categorical_cols:
    transformers.append(("categorical",Pipeline([
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]),categorical_cols))

preprocessor = ColumnTransformer(transformers,remainder="drop")
X_arr = preprocessor.fit_transform(df_raw[candidate_cols])
feature_names = preprocessor.get_feature_names_out()
X = pd.DataFrame(X_arr,columns=feature_names,index=df_raw.index).replace([np.inf,-np.inf],np.nan).fillna(0)
print("ID fields excluded:",identifier_cols)
print("Numeric:",numeric_cols)
print("Categorical:",categorical_cols)
print("Model matrix:",X.shape)
display(X.head())


## 4. Anomaly detection
Isolation Forest identifies records that are relatively easy to isolate in feature space. The contamination setting is an assumption; test alternatives and validate flagged records with domain knowledge.

In [ ]:

if len(X)<10:
    raise ValueError("At least 10 rows are recommended.")
iso = IsolationForest(n_estimators=300,contamination=ANOMALY_CONTAMINATION,
                      random_state=RANDOM_STATE,n_jobs=-1)
labels = iso.fit_predict(X)
scores = -iso.decision_function(X)
results = df_raw.copy()
results["anomaly_flag"] = np.where(labels==-1,"Potential anomaly","Typical")
results["anomaly_score"] = scores
anomalies = results[results.anomaly_flag=="Potential anomaly"].sort_values("anomaly_score",ascending=False)
print("Flagged:",len(anomalies),f"({len(anomalies)/len(results):.2%})")
display(anomalies.head(15))
anomalies.to_csv(OUTPUT_DIR/"02_potential_anomalies.csv",index=False)


## 5. Cluster discovery
Evaluate candidate K-Means solutions with silhouette score. This is a mathematical heuristic, not proof that groups have real-world meaning.

In [ ]:

score_rows = []
for k in range(2,min(9,len(X))):
    model = KMeans(n_clusters=k,random_state=RANDOM_STATE,n_init=15)
    lab = model.fit_predict(X)
    if len(np.unique(lab))>1:
        score = silhouette_score(X,lab,sample_size=min(2500,len(X)),random_state=RANDOM_STATE)
        score_rows.append({"k":k,"silhouette":score})
score_table = pd.DataFrame(score_rows)
if score_table.empty:
    raise ValueError("Unable to evaluate clustering.")
best_k = int(score_table.loc[score_table.silhouette.idxmax(),"k"])
kmeans = KMeans(n_clusters=best_k,random_state=RANDOM_STATE,n_init=20)
results["discovered_cluster"] = kmeans.fit_predict(X)
print("Selected k:",best_k)
display(score_table.sort_values("silhouette",ascending=False))
score_table.to_csv(OUTPUT_DIR/"03_cluster_model_selection.csv",index=False)


## 6. PCA latent structure
PCA projects the transformed features into two dimensions. Axes are combinations of features, not original measurements.

In [ ]:

pca = PCA(n_components=2,random_state=RANDOM_STATE)
coords = pca.fit_transform(X)
results["pca_1"],results["pca_2"] = coords[:,0],coords[:,1]
pca_variance = pca.explained_variance_ratio_
print("PC1:",f"{pca_variance[0]:.2%}","PC2:",f"{pca_variance[1]:.2%}",
      "Combined:",f"{pca_variance.sum():.2%}")


## 7. Relationship discovery
Spearman correlation measures monotonic association. Pairwise p-values are unadjusted and exploratory; multiple comparisons and confounding can mislead.

In [ ]:

usable_numeric = [c for c in numeric_cols if df_raw[c].notna().sum()>=5 and df_raw[c].nunique(dropna=True)>1]
numeric_data = df_raw[usable_numeric].replace([np.inf,-np.inf],np.nan)
corr = numeric_data.corr(method="spearman")
rows = []
for i,a in enumerate(usable_numeric):
    for b in usable_numeric[i+1:]:
        pair = numeric_data[[a,b]].dropna()
        if len(pair)>=5:
            rho,pval = spearmanr(pair[a],pair[b])
            rows.append({"feature_1":a,"feature_2":b,"spearman_rho":rho,
                         "p_value_unadjusted":pval,"n_pairwise":len(pair),
                         "absolute_strength":abs(rho)})
relationships = pd.DataFrame(rows)
if not relationships.empty:
    relationships = relationships.sort_values("absolute_strength",ascending=False)
    display(relationships.head(20))
    relationships.to_csv(OUTPUT_DIR/"04_pairwise_relationships.csv",index=False)
else:
    print("Not enough numeric pairs.")


## 8. Mutual-information signals
For each numeric reference target, estimate mutual information from the remaining numeric features. This is an exploratory dependence measure, not causal importance.

In [ ]:

mi_rows = []
if len(usable_numeric)>=2:
    mi_data = numeric_data.fillna(numeric_data.median(numeric_only=True))
    for target in usable_numeric:
        predictors = [c for c in usable_numeric if c!=target]
        scores_mi = mutual_info_regression(mi_data[predictors].to_numpy(),
                                           mi_data[target].to_numpy(),
                                           random_state=RANDOM_STATE)
        for feature,score in zip(predictors,scores_mi):
            mi_rows.append({"reference_target":target,"feature":feature,
                            "mutual_information":float(score)})
mi_results = pd.DataFrame(mi_rows)
if not mi_results.empty:
    mi_summary = mi_results.groupby("feature").mutual_information.mean().sort_values(ascending=False).reset_index()
    display(mi_summary)
    mi_results.to_csv(OUTPUT_DIR/"05_mutual_information_pairings.csv",index=False)
    mi_summary.to_csv(OUTPUT_DIR/"06_mutual_information_summary.csv",index=False)
else:
    mi_summary = pd.DataFrame()
    print("At least two numeric features are needed.")


## 9. Visual investigation dashboard
Use these plots to guide follow-up questions. They are exploratory summaries.

In [ ]:

# Missingness profile
missing_pct = df_raw.isna().mean().mul(100).sort_values(ascending=False)
plt.figure(figsize=(12,5)); sns.barplot(x=missing_pct.index,y=missing_pct.values)
plt.xticks(rotation=55,ha="right"); plt.ylabel("Missing (%)")
plt.title("DATAMYST | Missingness by column"); plt.tight_layout(); plt.show()

# Correlation heatmap
if len(usable_numeric)>=2:
    cols = usable_numeric[:MAX_HEATMAP_FEATURES]
    plt.figure(figsize=(max(9,len(cols)*.65),max(7,len(cols)*.55)))
    sns.heatmap(corr.loc[cols,cols],cmap="vlag",center=0,
                annot=len(cols)<=12,fmt=".2f",linewidths=.3)
    plt.title("Spearman relationship map"); plt.tight_layout(); plt.show()

# Anomaly scores
plt.figure(figsize=(10,5)); sns.histplot(results.anomaly_score,bins=35,kde=True)
plt.title("Isolation Forest anomaly-score distribution")
plt.xlabel("Anomaly score (higher = more unusual)"); plt.tight_layout(); plt.show()

# Cluster selection
plt.figure(figsize=(8,4)); sns.lineplot(data=score_table,x="k",y="silhouette",marker="o")
plt.title("K-Means model selection"); plt.tight_layout(); plt.show()

# PCA map
plt.figure(figsize=(10,7))
sns.scatterplot(data=results,x="pca_1",y="pca_2",hue="discovered_cluster",
                style="anomaly_flag",palette="tab10",alpha=.75,s=45)
plt.title("Latent structure: clusters and anomaly flags"); plt.tight_layout(); plt.show()

# Strongest associations
if not relationships.empty:
    top = relationships.head(12).copy()
    top["pair"] = top.feature_1+" ↔ "+top.feature_2
    plt.figure(figsize=(11,6))
    sns.barplot(data=top,x="spearman_rho",y="pair",color="slateblue")
    plt.axvline(0,color="black",lw=.8)
    plt.title("Strongest Spearman associations"); plt.tight_layout(); plt.show()

# Cluster sizes
plt.figure(figsize=(8,4))
sns.countplot(data=results,x="discovered_cluster",hue="discovered_cluster",
              palette="viridis",legend=False)
plt.title("Records per discovered cluster"); plt.tight_layout(); plt.show()

# PCA explained variance
plt.figure(figsize=(7,4)); plt.bar(["PC1","PC2"],pca_variance*100)
plt.ylabel("Explained variance (%)"); plt.title("PCA variance retained")
plt.tight_layout(); plt.show()


## 10. Cluster profiles
Compare original numeric fields across discovered groups. These descriptions require domain interpretation.

In [ ]:

if numeric_cols:
    cluster_profiles = results.groupby("discovered_cluster")[numeric_cols].agg(["mean","median","std"]).round(3)
    display(cluster_profiles)
    cluster_profiles.to_csv(OUTPUT_DIR/"07_cluster_profiles.csv")
display(results["discovered_cluster"].value_counts().sort_index().rename("record_count").to_frame())


## 11. Ranked discovery register
A compact evidence register combines association and information-signal findings without mixing incomparable metrics into one score.

In [ ]:

discovery_rows = []
if not relationships.empty:
    for _,r in relationships.head(15).iterrows():
        discovery_rows.append({"type":"Pairwise association",
            "discovery":f"{r.feature_1} ↔ {r.feature_2}",
            "metric":"Spearman rho","value":r.spearman_rho,
            "unadjusted_p":r.p_value_unadjusted,"n":r.n_pairwise})
if not mi_summary.empty:
    for _,r in mi_summary.head(10).iterrows():
        discovery_rows.append({"type":"Information signal",
            "discovery":r.feature,"metric":"Mean mutual information",
            "value":r.mutual_information,"unadjusted_p":np.nan,"n":len(results)})
discovery_register = pd.DataFrame(discovery_rows)
display(discovery_register.head(25))
discovery_register.to_csv(OUTPUT_DIR/"08_ranked_discovery_register.csv",index=False)


## 12. Automated investigation report and exports

In [ ]:

report = [
    "DATAMYST AI — DATA FORENSICS REPORT","="*60,
    f"Dataset source: {data_source}",f"Rows: {len(df_raw):,}",
    f"Columns: {df_raw.shape[1]:,}",f"Duplicate rows: {duplicate_count:,}",
    f"Missing cells: {int(df_raw.isna().sum().sum()):,}",
    f"Heuristic quality score: {quality_score:.1f}/100",
    f"Potential anomalies: {len(anomalies):,}",
    f"Selected cluster count: {best_k}",
    f"Two-component PCA variance: {pca_variance.sum():.2%}",
    "","TOP ASSOCIATIONS"
]
if not relationships.empty:
    for _,r in relationships.head(5).iterrows():
        report.append(f"- {r.feature_1} vs {r.feature_2}: rho={r.spearman_rho:.3f}; unadjusted p={r.p_value_unadjusted:.4g}")
else:
    report.append("- No eligible numeric feature pairs.")
report += ["","LIMITATIONS",
    "- Anomaly flags are review candidates, not proof of wrongdoing.",
    "- Associations do not establish causation.",
    "- Unadjusted p-values are exploratory; multiple testing may inflate false positives.",
    "- Clusters are algorithmic and require domain interpretation.",
    "- Synthetic demonstration data must not be reported as real-world findings."]
report_text = "\n".join(report)
print(report_text)
(OUTPUT_DIR/"09_DATAMYST_investigation_report.txt").write_text(report_text,encoding="utf-8")

results.to_csv(OUTPUT_DIR/"10_DATAMYST_full_results.csv",index=False)
manifest = {
    "project":"DATAMYST AI","source":data_source,
    "rows":int(len(df_raw)),"columns":int(df_raw.shape[1]),
    "quality_score_heuristic":round(quality_score,3),
    "anomaly_count":int(len(anomalies)),"selected_cluster_count":int(best_k),
    "pca_two_component_variance":round(float(pca_variance.sum()),6)
}
(OUTPUT_DIR/"11_run_manifest.json").write_text(json.dumps(manifest,indent=2),encoding="utf-8")

print("\\nCompleted. Output folder:",OUTPUT_DIR.resolve())
for f in sorted(OUTPUT_DIR.iterdir()):
    print(" -",f.name)


## 13. Research-grade next steps

- Add domain-specific validation rules (valid ranges, date logic, key uniqueness).
- Compare Isolation Forest with Local Outlier Factor and robust covariance.
- Correct for multiple statistical tests (for example, Benjamini–Hochberg false-discovery-rate control).
- Test cluster stability across random seeds and bootstrap samples.
- Add provenance, schema versioning, and reproducible run metadata.
- If building a supervised model, define a defensible target and evaluate on a held-out test set.
- Use ethically sourced, licensed data and document privacy protections.

**Portfolio positioning:** an explainable investigation workflow that turns raw tables into traceable quality findings, anomaly candidates, latent groups, and ranked statistical signals.